# sPHENIX Group Notebook: Dijet Momentum Balance
## Finding the Critical Point: The xJ Observable

**Vanderbilt Programs for Talented Youth | Instructor: Jennifer James, Ph.D. Candidate**

---

### What is xJ?

When two high-energy quarks or gluons collide inside a proton-proton or gold-gold collision, they fly apart back-to-back as **jets** (showers of particles we discussed earlier this week). In vacuum (p+p), the two jets should be roughly balanced in momentum.

In a QGP (Au+Au), one jet may have to travel through more hot matter than the other. It loses energy to the medium, and comes out softer. The **dijet momentum balance**:

$$x_J = \frac{p_{T,2}}{p_{T,1}}$$

where $p_{T,1}$ is the **leading** (harder) jet and $p_{T,2}$ is the **subleading** (softer) jet, tells us how imbalanced the pair is.

- **xJ = 1** → perfectly balanced, no quenching
- **xJ → 0** → subleading jet severely quenched

In p+p, xJ peaks near 1. In central Au+Au, the distribution shifts toward lower xJ, meaning the subleading jet lost energy in the QGP.

This is one of the key measurements sPHENIX was built to make precisely, and it connects directly to understanding **how** energy is lost in the QGP. This is related to the search for the critical point of the QCD phase diagram.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)
print("Libraries loaded!")

---

## Part 1: Generating Dijet Events

We simulate dijet events using a toy model. The physics ingredients are:

1. **Leading jet pT** drawn from a power-law spectrum (real jets follow this)
2. **Subleading jet pT** in p+p: smeared around xJ = 1 (balanced, with detector resolution)
3. **Subleading jet pT** in Au+Au: shifted to lower xJ (quenching) and broadened (path-length fluctuations)

The amount of quenching depends on **centrality** (more central = more QGP = more quenching).

In [ ]:
# ── Simulation parameters ─────────────────────────────────────────────────────
N_EVENTS   = 50000       # events per system
PT1_MIN    = 20.0        # minimum leading jet pT (GeV/c)
PT1_MAX    = 80.0        # maximum leading jet pT (GeV/c)
POWER      = -5.0        # jet spectrum power law index

def sample_leading_pT(n, pt_min=PT1_MIN, pt_max=PT1_MAX, alpha=POWER):
    """Sample leading jet pT from a power-law distribution."""
    u = np.random.uniform(0, 1, n)
    exp = alpha + 1
    return (pt_min**exp + u * (pt_max**exp - pt_min**exp))**(1/exp)

def sample_xJ(n, mean_xJ, sigma_xJ, xJ_min=0.3):
    """Sample xJ from a truncated Gaussian."""
    xJ = np.random.normal(mean_xJ, sigma_xJ, n)
    # Truncate: xJ must be in [xJ_min, 1.0]
    xJ = np.clip(xJ, xJ_min, 1.0)
    return xJ

# ── p+p reference ──────────────────────────────────────────────────────────────
pt1_pp   = sample_leading_pT(N_EVENTS)
xJ_pp    = sample_xJ(N_EVENTS, mean_xJ=0.92, sigma_xJ=0.08)
pt2_pp   = pt1_pp * xJ_pp

# ── Au+Au central (0-10%): strong quenching ────────────────────────────────────
pt1_central = sample_leading_pT(N_EVENTS)
xJ_central  = sample_xJ(N_EVENTS, mean_xJ=0.70, sigma_xJ=0.18)
pt2_central = pt1_central * xJ_central

# ── Au+Au mid-central (20-40%): moderate quenching ───────────────────────────
pt1_mid = sample_leading_pT(N_EVENTS)
xJ_mid  = sample_xJ(N_EVENTS, mean_xJ=0.82, sigma_xJ=0.13)
pt2_mid = pt1_mid * xJ_mid

# ── Au+Au peripheral (60-80%): mild quenching ────────────────────────────────
pt1_periph = sample_leading_pT(N_EVENTS)
xJ_periph  = sample_xJ(N_EVENTS, mean_xJ=0.89, sigma_xJ=0.09)
pt2_periph = pt1_periph * xJ_periph

print(f"Generated {N_EVENTS:,} dijet events per system")
print(f"  p+p:           mean xJ = {xJ_pp.mean():.3f},  std = {xJ_pp.std():.3f}")
print(f"  Au+Au central: mean xJ = {xJ_central.mean():.3f},  std = {xJ_central.std():.3f}")
print(f"  Au+Au mid:     mean xJ = {xJ_mid.mean():.3f},  std = {xJ_mid.std():.3f}")
print(f"  Au+Au periph:  mean xJ = {xJ_periph.mean():.3f},  std = {xJ_periph.std():.3f}")

---

## Part 2: The xJ Distribution: Seeing Quenching

In [ ]:
bins = np.linspace(0.3, 1.0, 29)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: all four systems overlaid ──────────────────────────────────────────
ax = axes[0]
for data, label, color, ls in [
    (xJ_pp,       'p+p (reference)',          'black',      '-'),
    (xJ_periph,   'Au+Au peripheral (60-80%)', 'steelblue',  '--'),
    (xJ_mid,      'Au+Au mid-central (20-40%)','darkorange', '-.'),
    (xJ_central,  'Au+Au central (0-10%)',     'firebrick',  '-'),
]:
    counts, _ = np.histogram(data, bins=bins, density=True)
    ax.plot(bin_centers, counts, color=color, linewidth=2.5,
            linestyle=ls, label=label)

ax.axvline(x=1.0, color='gray', linestyle=':', linewidth=1.2, alpha=0.7)
ax.set_xlabel(r'$x_J = p_{T,2} / p_{T,1}$', fontsize=13)
ax.set_ylabel('Normalized yield', fontsize=13)
ax.set_title('Dijet momentum balance $x_J$\nAll centralities', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.3, 1.05)

# ── Right: p+p vs central only, with suppression shaded ──────────────────────
ax = axes[1]
counts_pp, _  = np.histogram(xJ_pp,      bins=bins, density=True)
counts_c,  _  = np.histogram(xJ_central, bins=bins, density=True)

ax.plot(bin_centers, counts_pp, color='black',    linewidth=2.5, label='p+p (reference)')
ax.plot(bin_centers, counts_c,  color='firebrick', linewidth=2.5, label='Au+Au central (0-10%)')

ax.fill_between(bin_centers, counts_pp, counts_c,
                where=(counts_pp > counts_c), alpha=0.25, color='steelblue',
                label='Excess in p+p (quenched away in Au+Au)')
ax.fill_between(bin_centers, counts_pp, counts_c,
                where=(counts_c > counts_pp), alpha=0.25, color='firebrick',
                label='Excess in Au+Au (energy redistributed to low xJ)')

ax.axvline(x=1.0, color='gray', linestyle=':', linewidth=1.2, alpha=0.7)
ax.set_xlabel(r'$x_J = p_{T,2} / p_{T,1}$', fontsize=13)
ax.set_ylabel('Normalized yield', fontsize=13)
ax.set_title('p+p vs central Au+Au\nWhere does the subleading jet energy go?', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.3, 1.05)

plt.suptitle('Dijet Momentum Balance in Heavy-Ion Collisions\n'
             'Toy model inspired by sPHENIX / CMS measurements', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("The p+p distribution peaks near xJ = 1: balanced dijets.")
print("Central Au+Au shifts to lower xJ: the subleading jet lost energy in the QGP.")
print("The lost energy doesn't disappear, rather, it gets redistributed to soft particles.")

---

## Part 3: Mean xJ vs Centrality: Quantifying the Quenching

In [ ]:
# Mean xJ as a function of centrality
systems    = ['p+p\n(reference)', 'Peripheral\n(60-80%)', 'Mid-central\n(20-40%)', 'Central\n(0-10%)']
mean_xJs   = [xJ_pp.mean(), xJ_periph.mean(), xJ_mid.mean(), xJ_central.mean()]
err_xJs    = [xJ_pp.std()/np.sqrt(N_EVENTS)*5,   # scaled for visibility
              xJ_periph.std()/np.sqrt(N_EVENTS)*5,
              xJ_mid.std()/np.sqrt(N_EVENTS)*5,
              xJ_central.std()/np.sqrt(N_EVENTS)*5]

colors = ['black', 'steelblue', 'darkorange', 'firebrick']

fig, ax = plt.subplots(figsize=(8, 5))
x_pos = np.arange(len(systems))
bars = ax.bar(x_pos, mean_xJs, color=colors, edgecolor='black',
              linewidth=0.8, alpha=0.85, yerr=err_xJs, capsize=5)
ax.axhline(y=mean_xJs[0], color='black', linestyle='--', linewidth=1.5,
           alpha=0.5, label=f'p+p reference: <xJ> = {mean_xJs[0]:.3f}')
ax.set_xticks(x_pos)
ax.set_xticklabels(systems, fontsize=11)
ax.set_ylabel(r'Mean dijet momentum balance $\langle x_J angle$', fontsize=12)
ax.set_title('Centrality dependence of dijet quenching\n'
             'More central = more QGP = more momentum imbalance', fontsize=12)
ax.set_ylim(0.5, 1.05)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, mean_xJs):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.008,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Quenching effect: central Au+Au mean xJ is")
print(f"  {(mean_xJs[0]-mean_xJs[-1])/mean_xJs[0]*100:.1f}% lower than p+p reference.")
print()
print("Question: Why does more central = more quenching?")
print("Question: Does the subleading or leading jet lose more energy? Why?")

---

## Part 4: Connection to the Critical Point

sPHENIX's jet measurements connect to the search for the **QCD critical point**, the endpoint of the first-order phase transition line between hadronic matter and the QGP on the temperature-baryon density phase diagram.

Near the critical point, the medium's response to an energy-depositing jet would change dramatically. The **path-length dependence** of energy loss (how much a jet loses per unit distance through the medium), is sensitive to the medium's properties near the critical point.

By measuring xJ across many beam energies (the RHIC Beam Energy Scan program), physicists look for non-monotonic changes in quenching that would signal the critical point.

In [ ]:
# Toy: how xJ mean shifts with beam energy near the critical point
# The critical point is hypothesized around sqrt(s_NN) ~ 20-30 GeV
# Near it, quenching would be enhanced due to large fluctuations

sqrt_s = np.array([7.7, 11.5, 14.5, 19.6, 27.0, 39.0, 62.4, 130.0, 200.0])

# Far from critical point: smooth variation
mean_xJ_smooth = 0.68 + 0.14 * (1 - np.exp(-sqrt_s / 50))

# Near critical point (~20 GeV): extra suppression dip
critical_dip = 0.06 * np.exp(-0.5 * ((np.log(sqrt_s) - np.log(20))/0.4)**2)
mean_xJ_critical = mean_xJ_smooth - critical_dip

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sqrt_s, mean_xJ_smooth,   'k--', linewidth=2, label='Smooth (no critical point)', alpha=0.6)
ax.plot(sqrt_s, mean_xJ_critical, 'firebrick', linewidth=2.5, marker='o', markersize=8,
        label='With critical point signal (toy)')
ax.fill_between(sqrt_s, mean_xJ_smooth, mean_xJ_critical,
                alpha=0.25, color='firebrick', label='Critical point signature')
ax.axvspan(15, 30, alpha=0.08, color='gold', label='Hypothesized critical point region')
ax.set_xscale('log')
ax.set_xlabel(r'$\sqrt{s_{NN}}$ (GeV) (beam energy)', fontsize=13)
ax.set_ylabel(r'Mean $\langle x_J \rangle$ in central Au+Au', fontsize=13)
ax.set_title('Toy model: Beam Energy Scan (searching for the QCD critical point)\n'
             'A dip in <xJ> near the critical point energy would be a signal', fontsize=11)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("Real sPHENIX data will map out this curve precisely.")
print("A non-monotonic dip in jet quenching vs beam energy")
print("would be one of the strongest signals for the QCD critical point.")

---

## Your Group's Checkpoint Questions

1. In your xJ plot, which centrality class shows the most balanced dijets? Which shows the most imbalanced? Does this match your expectation from R_AA?

2. The subleading jet loses more energy than the leading jet on average. Why? *(Hint: think about which jet is more likely to be produced near the surface of the QGP fireball vs. deep inside it.)*

3. In the beam energy scan plot, what does the dip near 20 GeV represent physically? What property of the QGP medium would change near the critical point to cause extra quenching?

4. sPHENIX measures jet **substructure** (Rg, zg) in addition to xJ. Why isn't xJ alone enough to fully characterize how a jet interacts with the QGP?